# 07 — Loss diagnosis on `pretend-2024`

Reproduces the per-rm loss table that drove the v4 design. Top 25 rm_ids contribute ~96.5% of total pinball loss; the design targets that concentration.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.data import load_or_build, build_profile, cumulative_truth
from src.gating import assign_tracks
from src.regime import classify_regime
from src.models.linear_per_rm import PerRMLinearForecaster
from src.metric import pinball_loss
from src.validation import DEFAULT_FOLDS, build_query_for_fold


In [ ]:
ds = load_or_build()
all_rm_ids = sorted(ds.daily['rm_id'].unique().tolist())
fold = DEFAULT_FOLDS[1]   # pretend-2024
cutoff = fold.train_end + pd.Timedelta(days=1)
daily_pre = ds.daily[ds.daily['date'] < cutoff]
profile = build_profile(daily_pre, year=fold.target_year - 1)
tracks = assign_tracks(profile, all_rm_ids=all_rm_ids)
regimes = classify_regime(ds.daily, cutoff=cutoff)
track_a = set(tracks[tracks['track']=='A']['rm_id'])
track_b = set(tracks[tracks['track']=='B']['rm_id'])
track_c = set(tracks[tracks['track']=='C']['rm_id'])
intermittent = set(regimes[regimes['regime']=='INTERMITTENT']['rm_id'])

per_rm = {**{rm: 0.7 for rm in track_a}, **{rm: 0.7 for rm in track_b}, **{rm: 0.3 for rm in track_c}}
for rm in list(per_rm):
    if rm in intermittent: del per_rm[rm]

m = PerRMLinearForecaster(fit_year=fold.target_year-1, slope_strategy='trailing_window', trailing_window_days=210, cutoff=cutoff, slope_shrink=1.0).fit(ds.daily)
preds = m.predict(build_query_for_fold(fold, all_rm_ids), rm_id_track_filter=set(per_rm), per_rm_shrink=per_rm)
truth = cumulative_truth(ds.daily, fold.target_year)
merged = preds.merge(truth, on=['rm_id','forecast_end_date'], how='left').fillna(0)
merged['loss'] = pinball_loss(merged['predicted_weight'].to_numpy(), merged['actual_weight'].to_numpy())
print('Total mean pinball:', merged['loss'].mean())


In [ ]:
per = merged.groupby('rm_id').agg(loss=('loss','mean'), pred=('predicted_weight','last'), actual=('actual_weight','last')).reset_index()
per['ratio'] = per['pred'] / per['actual'].replace(0, np.nan)
per['regime'] = per['rm_id'].map(dict(zip(regimes['rm_id'], regimes['regime'])))
per['track']  = per['rm_id'].map(dict(zip(tracks['rm_id'], tracks['track'])))
per['contrib_pct'] = (per['loss'] * 150 / 30450 / merged['loss'].mean() * 100).round(2)
top = per.sort_values('loss', ascending=False).head(25)
print(f'Top 25 contribute {top.contrib_pct.sum():.1f}% of total loss')
top[['rm_id','regime','track','pred','actual','ratio','contrib_pct']]


## Regime distribution
Most rm_ids should be INTERMITTENT (no recent activity); the active cohort is in DEFAULT/STABLE/GROWING/DECLINING/NEW.

In [ ]:
regimes['regime'].value_counts()
